In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU detected: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("❌ No GPU detected, running on CPU")


2026-01-05 18:40:40.243896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767638440.440334      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767638440.494493      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767638440.978203      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767638440.978265      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767638440.978268      55 computation_placer.cc:177] computation placer alr

✅ GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Core
import numpy as np
import pandas as pd

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [5]:
import os

os.listdir("/kaggle/input/ember-feature-based-dataset/ember")


['test_ember_2018_final.parquet',
 'train_ember_2018_final.parquet',
 'unlabelled_set_for_pred.parquet']

In [6]:
import pandas as pd

train_df = pd.read_parquet(
    "/kaggle/input/ember-feature-based-dataset/ember/train_ember_2018_final.parquet"
)

test_df = pd.read_parquet(
    "/kaggle/input/ember-feature-based-dataset/ember/test_ember_2018_final.parquet"
)

print(train_df.shape)
print(test_df.shape)

(599920, 2342)
(199956, 2342)


In [7]:
# Shuffle training data
train_df = train_df.sample(frac=1, random_state=42)

# Client-1 gets first 50% of TRAIN data
client1_df = train_df.iloc[: int(0.5 * len(train_df))]

print("Client-1 shape:", client1_df.shape)

Client-1 shape: (299960, 2342)


In [11]:
client1_df.columns

Index(['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10',
       ...
       'F2373', 'F2374', 'F2375', 'F2376', 'F2377', 'F2378', 'F2379', 'F2380',
       'F2381', 'Label'],
      dtype='object', length=2342)

In [12]:
X_train = client1_df.drop(columns=["Label"]).values
y_train = client1_df["Label"].values

X_test = test_df.drop(columns=["Label"]).values
y_test = test_df["Label"].values

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1767640141.075316      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     1,199,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,330,689 (5.08 MB)

 Trainable params: 1,330,689 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=512,   # good for P100
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20


I0000 00:00:1767640187.794802     215 service.cc:152] XLA service 0x49cf6550 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767640187.794867     215 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767640188.166083     215 cuda_dnn.cc:529] Loaded cuDNN version 91002


 35/528 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7585 - loss: 0.4974

I0000 00:00:1767640190.019491     215 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


528/528 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8834 - loss: 0.2725 - val_accuracy: 0.9377 - val_loss: 0.1515
Epoch 2/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9407 - loss: 0.1440 - val_accuracy: 0.9464 - val_loss: 0.1308
Epoch 3/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9502 - loss: 0.1212 - val_accuracy: 0.9495 - val_loss: 0.1278
Epoch 4/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9560 - loss: 0.1075 - val_accuracy: 0.9549 - val_loss: 0.1233
Epoch 5/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9599 - loss: 0.0983 - val_accuracy: 0.9538 - val_loss: 0.1288
Epoch 6/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9630 - loss: 0.0913 - val_accuracy: 0.9559 - val_loss: 0.1277
Epoch 7/20
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9657 - loss: 0.0839 - val_accuracy: 0.9573 - val_loss: 0.1264


In [16]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

y_pred = (model.predict(X_test) > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_pred)

print("Test Accuracy:", acc)
print("Test F1:", f1)
print("Test ROC-AUC:", roc)

6249/6249 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
Test Accuracy: 0.9380313669007182
Test F1: 0.938470476653938
Test ROC-AUC: 0.9380318711689279


In [18]:
model.save("ember_client1.keras")